# 05  Python Modeling and Evaluation

In [22]:
pip install scikit-learn

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [23]:
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (roc_auc_score, average_precision_score, precision_score, recall_score,
                             f1_score, confusion_matrix, precision_recall_curve)
from sklearn.inspection import permutation_importance

In [24]:
SEED = 42
DATA = "C:/Users/Siddhesh/Downloads/bank-customer-churn/data/raw/bank_churn.csv"
raw = pd.read_csv(DATA)
raw.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [25]:
assert raw["CustomerId"].is_unique
ids = raw["CustomerId"]
y = raw["Exited"].astype(int)
df = raw.drop(columns=["RowNumber", "CustomerId", "Surname", "Exited"])

In [26]:
df["HasZeroBalance"]     = (df["Balance"] == 0).astype(int)
df["BalanceToSalary"]    = df["Balance"] / df["EstimatedSalary"].replace(0, np.nan)
df["AgeBand"]            = pd.cut(df["Age"], [0,29,39,49,59,120],
                                  labels=["<30","30-39","40-49","50-59","60+"]).astype(str)
df["ProductsGroup"]      = df["NumOfProducts"].clip(upper=3).astype(str).replace({"3": "3+"})
df["TenureYoungAccount"] = (df["Tenure"] <= 1).astype(int)

In [27]:
X_tr, X_te, y_tr, y_te = train_test_split(df, y, test_size=0.2, stratify=y, random_state=SEED)

In [28]:
def build_models(num, cat):
    def prep(scale):
        steps = [("imp", SimpleImputer(strategy="median"))] + ([("sc", StandardScaler())] if scale else [])
        return ColumnTransformer([("num", Pipeline(steps), num),
                                  ("cat", OneHotEncoder(handle_unknown="ignore"), cat)])
    return { "Baseline (majority)": ..., "LogReg": ..., "RandomForest": ..., "HistGB": ... }

num = X_tr.select_dtypes("number").columns.tolist()
cat = X_tr.select_dtypes(exclude="number").columns.tolist()
models = build_models(num, cat)

In [ ]:
cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
rows = []
for name, pipe in models.items():
    def build_models(num, cat):
        def prep(scale=False):
            numeric_steps = [("imp", SimpleImputer(strategy="median"))]
            if scale:
                numeric_steps.append(("sc", StandardScaler()))

            return ColumnTransformer(
                [
                    ("num", Pipeline(numeric_steps), num),
                    ("cat", OneHotEncoder(handle_unknown="ignore"), cat),
                ],
                # Produce dense output so HistGradientBoostingClassifier can use it.
                sparse_threshold=0,
            )

        return {
            "Baseline (majority)": DummyClassifier(strategy="most_frequent"),
            "LogReg": Pipeline([
                ("prep", prep(scale=True)),
                ("clf", LogisticRegression(max_iter=1000, random_state=SEED)),
            ]),
            "RandomForest": Pipeline([
                ("prep", prep()),
                ("clf", RandomForestClassifier(
                    n_estimators=300, random_state=SEED, n_jobs=-1
                )),
            ]),
            "HistGB": Pipeline([
                ("prep", prep()),
                ("clf", HistGradientBoostingClassifier(random_state=SEED)),
            ]),
        }

    num = X_tr.select_dtypes("number").columns.tolist()
    cat = X_tr.select_dtypes(exclude="number").columns.tolist()
    models = build_models(num, cat)
    rows.append({
        "model": name,
        "ROC_AUC": round(s["test_roc_auc"].mean(), 3),
        "PR_AUC": round(s["test_average_precision"].mean(), 3),
        "Recall": round(s["test_recall"].mean(), 3),
        "Precision": round(s["test_precision"].mean(), 3),
        "PR_AUC_STD": round(s["test_average_precision"].std(), 3)
    })
cmp_df = pd.DataFrame(rows).set_index("model"); print(cmp_df)

InvalidParameterError: The 'estimator' parameter of cross_validate must be an object implementing 'fit'. Got Ellipsis instead.

In [30]:
print(models)

{'Baseline (majority)': Ellipsis, 'LogReg': Ellipsis, 'RandomForest': Ellipsis, 'HistGB': Ellipsis}


In [31]:
models = build_models(num, cat)

for name, pipe in models.items():
    print(name, type(pipe))

Baseline (majority) <class 'ellipsis'>
LogReg <class 'ellipsis'>
RandomForest <class 'ellipsis'>
HistGB <class 'ellipsis'>


In [21]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED
)

rows = []

for name, pipe in models.items():
    s = cross_validate(
        pipe,
        X_tr,
        y_tr,
        cv=cv,
        scoring=["roc_auc", "average_precision", "recall", "precision"],
    )

    rows.append({
        "model": name,
        "ROC_AUC": round(s["test_roc_auc"].mean(), 3),
        "PR_AUC": round(s["test_average_precision"].mean(), 3),
        "Recall": round(s["test_recall"].mean(), 3),
        "Precision": round(s["test_precision"].mean(), 3),
        "PR_AUC_STD": round(s["test_average_precision"].std(), 3)
    })

cmp_df = pd.DataFrame(rows).set_index("model")

print(cmp_df)

C:\Users\Siddhesh\AppData\Roaming\Python\Python314\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\Siddhesh\AppData\Roaming\Python\Python314\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\Siddhesh\AppData\Roaming\Python\Python314\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", res

                     ROC_AUC  PR_AUC  Recall  Precision  PR_AUC_STD
model                                                              
Baseline (majority)    0.500   0.204   0.000      0.000       0.000
LogReg                 0.843   0.652   0.421      0.727       0.020
RandomForest           0.850   0.669   0.454      0.735       0.023
HistGB                 0.856   0.689   0.486      0.724       0.028
